# Killer Sudoku Solver
# Project 3 - DSA 8420
# By: Saransh Rakshak
# Due: April 29, 2025


![Sudoku](sudoku_player.jpg)

## Killer Sudoku Solver With Pyomo

This notebook solves **killer sudoku** puzzles using Pyomo for mathematical optimization. The code supports arbitrary puzzle inputs and includes step by step instructions to modify and solve own puzzles.

The rules of **killer sudoku** are as follows:

1. Fill all rows, columns, and 3x3 blocks with numbers 1-9 exactly like in classic sudoku.

2. Pay attention to the cages – groups of cells indicated by dotted lines.

3. Make sure the sum of numbers in each cage is equal to the number in the upper left corner of the cage.

4. Numbers cannot repeat within cages, a single row, column, or 3x3 region.

## Step 1: Importing Libraries

Please import the following libraries. 

If not installed please install using **pip** (pip install numpy pyomo glpk) or **conda** (conda install -c conda-forge numpy pyomo glpk).

In [1]:
# import libraries
import numpy as np
import pyomo.environ as pyo
from pyomo.opt import SolverFactory

## Step 2: Defining Puzzle Inputs

Modify variables *known*, *cages*, and *cage_sums* to match custom killer sudoku puzzle.

- *known*: list of fixed cells, example: [(row, col, digit)]
- *cages*: list of cage groups with cell coordinates
- *cage_sums*: list of cage totals matching *cages* order


In [2]:
known = [(1, 7, 3), (2, 5, 3), (4, 2, 2), (5, 5, 2),
         (6, 7, 6), (7, 6, 1), (8, 3, 5), (8, 9, 2),
         (9, 4, 4), (9, 8, 3)]

cages = [[(1,1)], [(1,2), (1,3), (1,4)], [(1,5), (2,4), (2,5)],
    [(1,6), (2,6)], [(1,7), (1,8), (1,9), (2,8)], [(2,1)],
    [(2,2), (2,3)], [(2,7), (3,6), (3,7)], [(2,9)],
    [(3,1), (4,1)], [(3,2), (4,2), (5,1), (5,2), (6,1)],
    [(3,3), (4,3)], [(3,4), (3,5)], [(3,8), (3,9), (4,8)],
    [(4,4), (4,5)], [(4,6), (5,5), (5,6)], [(4,7), (5,7), (5,8)],
    [(4,9), (5,9)], [(5,3), (6,3)], [(5,4), (6,4), (6,5)],
    [(6,2), (7,2)], [(6,6), (6,7)], [(6,8), (7,7), (7,8)],
    [(6,9), (7,9), (8,9)], [(7,1), (8,1), (8,2)],
    [(7,3), (8,3), (8,4), (9,4)], [(7,4), (7,5), (7,6)],
    [(8,5), (9,5)], [(8,6), (9,6)],
    [(8,7), (8,8), (9,7), (9,8), (9,9)],
    [(9,1), (9,2)], [(9,3)]]

cage_sums = [9, 14, 11, 14, 18, 2, 12, 11, 6, 10, 17, 12, 13, 20, 9,
    18, 19, 4, 9, 14, 10, 10, 17, 15, 19, 17, 13, 17, 5, 24, 10, 6]

## Step 3: Creating Model and Variables

In [3]:
# pyomo model
model = pyo.ConcreteModel()

# sudoku grid
N = pyo.RangeSet(1,9)
# 3by3 block ind  
B = pyo.RangeSet(1,3)

# x[i,j,k]= 1 if i,j == k
model.x = pyo.Var(N,N,N, within= pyo.Binary)
# y[i,j] = digit in cell i,j
model.y = pyo.Var(N, N, within= pyo.Integers, bounds= (1,9))

## Step 4: Defining Objective Function

In [4]:
# no optim goal just solve
model.obj = pyo.Objective(expr= 0)

## Step 5: Establishing Sudoku Constraints

In [5]:
# one digit per cell
model.cell_unique = pyo.ConstraintList()
for i in N:
    for j in N:
        model.cell_unique.add(sum(model.x[i,j,k] for k in N)==1)

# one digit occurance in row
model.row_unique = pyo.ConstraintList()
for i in N:
    for k in N:
        model.row_unique.add(sum(model.x[i,j,k] for j in N)==1)

# one dig occurance in col
model.col_unique = pyo.ConstraintList()
for j in N:
    for k in N:
        model.col_unique.add(sum(model.x[i,j,k] for i in N)==1)

# one dig occurance in 3 by 3
model.block_unique = pyo.ConstraintList()
for block_row in range(0,9,3):
    for block_col in range(0,9,3):
        for k in N:
            model.block_unique.add(
                sum(model.x[i+1, j+1, k]
                    for i in range(block_row, block_row + 3)
                    for j in range(block_col, block_col + 3)
                ) == 1)

## Step 6: Link *x, y* variables.

- Refer to cell's value directly with y[i,j].

In [6]:
# y[i,j]=k only if x[i,j,k]==1
model.link = pyo.ConstraintList()
for i in N:
    for j in N:
        model.link.add(model.y[i,j] == sum(k * model.x[i,j,k] for k in N))

## Step 7: Establishing Known Values

In [7]:
model.known = pyo.ConstraintList()
for i, j, k in known:
    model.known.add(model.x[i,j,k] == 1)

## Step 8: Integrate Cage Constraints

In [8]:
# sum of num in each cage == upper left corner of cage
model.this_cage_sum = pyo.ConstraintList()
for ind, cage in enumerate(cages):
    model.this_cage_sum.add(sum(model.y[i,j] for i,j in cage) == cage_sums[ind])

# cant repeat within cages
model.cage_unique = pyo.ConstraintList()
for cage in cages:
    for k in N:
        model.cage_unique.add(sum(model.x[i,j,k] for i,j in cage) <= 1)

## Step 9: Solve Puzzle

In [9]:
# solving with glpk
solver = SolverFactory('glpk')
result = solver.solve(model, tee=True)

GLPSOL--GLPK LP/MIP Solver 5.0
Parameter(s) specified in the command line:
 --write C:\Users\sar-home\AppData\Local\Temp\tmplcwxna3a.glpk.raw --wglp
 C:\Users\sar-home\AppData\Local\Temp\tmpe4czu6uz.glpk.glp --cpxlp C:\Users\sar-home\AppData\Local\Temp\tmptgujxhof.pyomo.lp
Reading problem data from 'C:\Users\sar-home\AppData\Local\Temp\tmptgujxhof.pyomo.lp'...
C:\Users\sar-home\AppData\Local\Temp\tmptgujxhof.pyomo.lp:7655: warning: lower bound of variable 'x4' redefined
C:\Users\sar-home\AppData\Local\Temp\tmptgujxhof.pyomo.lp:7655: warning: upper bound of variable 'x4' redefined
735 rows, 811 columns, 4546 non-zeros
810 integer variables, 729 of which are binary
8384 lines were read
Writing problem data to 'C:\Users\sar-home\AppData\Local\Temp\tmpe4czu6uz.glpk.glp'...
6831 lines were written
GLPK Integer Optimizer 5.0
735 rows, 811 columns, 4546 non-zeros
810 integer variables, 729 of which are binary
Preprocessing...
504 rows, 452 columns, 2417 non-zeros
452 integer variables, 389 of

## Step 10: Display Solution

In [10]:
# pretty display of solution
solution = np.zeros((9,9), dtype= int)
for i in N:
    for j in N:
        solution[i-1, j-1]= int(pyo.value(model.y[i,j]))

print("Solved Killer Sudoku")
print("=====================")
print(solution)

Solved Killer Sudoku
[[9 8 4 2 7 6 3 1 5]
 [2 5 7 1 3 8 4 9 6]
 [6 1 3 9 4 5 2 7 8]
 [4 2 9 3 6 7 8 5 1]
 [5 6 1 8 2 9 7 4 3]
 [3 7 8 5 1 4 6 2 9]
 [8 3 2 7 5 1 9 6 4]
 [7 4 5 6 9 3 1 8 2]
 [1 9 6 4 8 2 5 3 7]]
